<a href="https://colab.research.google.com/github/mohith-hub/Medical-Abbreviation/blob/main/clg_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from tensorflow import keras
from packaging import version
import tensorflow as tf
import tensorflow_datasets as tfds
import tensorflow_hub as hub

In [ ]:
import sys
!{sys.executable} -m pip install Flask

In [19]:
!pip install pyngrok nest_asyncio

In [18]:
!pip install pyngrok nest_asyncio

In [ ]:
!pip install pyngrok nest_asyncio

In [13]:
import tensorflow_datasets as tfds
from sklearn.model_selection import train_test_split
import pandas as pd
import tensorflow as tf # Required for tfds.as_numpy_iterator() and other TF operations

# Load the IMDB reviews dataset from TensorFlow Datasets
# We load both train and test splits provided by TFDS.
# as_supervised=False returns examples as dictionaries {'text': ..., 'label': ...}
(ds_train_raw, ds_test_raw), info = tfds.load(
    name="imdb_reviews",
    split=["train", "test"],
    with_info=True,
    as_supervised=False
)

# Function to convert tf.data.Dataset to pandas DataFrame
def tf_dataset_to_df(tf_ds):
    reviews = []
    sentiments = []
    # Iterate through the dataset using as_numpy_iterator to get Python objects
    for example in tf_ds.as_numpy_iterator():
        reviews.append(example['text'].decode('utf-8')) # Decode byte string to regular string
        sentiments.append('positive' if example['label'] == 1 else 'negative')
    return pd.DataFrame({'review': reviews, 'sentiment': sentiments})

# Convert the raw TFDS datasets to pandas DataFrames
df_train_from_tfds = tf_dataset_to_df(ds_train_raw)
df_test_from_tfds = tf_dataset_to_df(ds_test_raw)

# Now, we align with the original notebook's variable names for subsequent cells.
# The original code created train, validation, and test sets from a single 'df'.
# With TFDS, we have a pre-defined train (df_train_from_tfds) and test (df_test_from_tfds).
# We will use df_train_from_tfds to create our 'train_set' and 'valid_set',
# and df_test_from_tfds will be our 'test_set'.

# Split the TFDS training set into training and validation sets (90/10 split)
train_set, valid_set = train_test_split(
    df_train_from_tfds,
    test_size=0.1,
    stratify=df_train_from_tfds['sentiment'],
    random_state=42 # for reproducibility
)

# Assign the TFDS test set as our final test_set
test_set = df_test_from_tfds

# The original `df` variable from the notebook is no longer directly populated in the same way.
# If `df` is used elsewhere, it might need to be defined (e.g., as a concatenation of train/test)
# or handled differently. Based on current context, only `train_set`, `valid_set`, `test_set` are used.

In [ ]:
def df_to_tf_dataset(df, batch_size=32, training=False):
    reviews = tf.constant(df['review'].values, dtype=tf.string)
    labels = tf.constant((df['sentiment'].values == 'positive').astype(int), dtype=tf.int64)

    ds = tf.data.Dataset.from_tensor_slices((reviews, labels))

    if training:
        ds = ds.shuffle(5000)

    ds = ds.batch(batch_size)
    ds = ds.prefetch(1)
    return ds


In [ ]:
train_set = df_to_tf_dataset(train_set, batch_size=32, training=True)
valid_set = df_to_tf_dataset(valid_set, batch_size=32)
test_set  = df_to_tf_dataset(test_set, batch_size=32)

In [ ]:
vocab_size = 1000
text_vec_layer = tf.keras.layers.TextVectorization(max_tokens=vocab_size)
text_vec_layer.adapt(train_set.map(lambda reviews, labels: reviews))


In [ ]:
embed_size = 128
tf.random.set_seed(42)
model = tf.keras.Sequential([
    text_vec_layer,
    tf.keras.layers.Embedding(vocab_size, embed_size),
    tf.keras.layers.GRU(128),
    tf.keras.layers.Dense(1, activation="sigmoid")
])
model.compile(loss="binary_crossentropy", optimizer="nadam",
              metrics=["accuracy"])
history = model.fit(train_set, validation_data=valid_set, epochs=2)

Epoch 1/2
 29/704 ━━━━━━━━━━━━━━━━━━━━ 13:46 1s/step - accuracy: 0.5035 - loss: 0.6957

KeyboardInterrupt: 

In [ ]:
embed_size = 128
tf.random.set_seed(42)
model = tf.keras.Sequential([
    text_vec_layer,
    tf.keras.layers.Embedding(vocab_size, embed_size,mask_zero = True),
    tf.keras.layers.GRU(128),
    tf.keras.layers.Dense(1, activation="sigmoid")
])
model.compile(loss="binary_crossentropy", optimizer="nadam",
              metrics=["accuracy"])
history = model.fit(train_set, validation_data=valid_set, epochs=5)

In [ ]:
text_vec_layer_ragged = tf.keras.layers.TextVectorization(
    max_tokens=vocab_size, ragged=True)
text_vec_layer_ragged.adapt(train_set.map(lambda reviews, labels: reviews))

In [ ]:
embed_size = 128
tf.random.set_seed(42)

model = tf.keras.Sequential([
    text_vec_layer,
    tf.keras.layers.Embedding(vocab_size, embed_size, mask_zero=True),
    tf.keras.layers.GRU(128),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(loss="binary_crossentropy", optimizer="nadam",
              metrics=["accuracy"])
history
= model.fit(train_set, validation_data=valid_set, epochs=5)

In [ ]:
import os
import tensorflow_hub as hub
from tensorflow import keras


os.environ["TFHUB_CACHE_DIR"] = "my_tfhub_cache"

class USELayer(keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.use_layer = hub.KerasLayer(
            "https://tfhub.dev/google/universal-sentence-encoder/4",
            trainable=True,
            name="USE_embedding"
        )

    def call(self, inputs):
        return self.use_layer(inputs)

use_layer = USELayer()

text_input = keras.layers.Input(shape=[], dtype="string", name="text")
embedding = use_layer(text_input)
dense = keras.layers.Dense(128, activation="relu")(embedding)
output = keras.layers.Dense(1, activation="sigmoid")(dense)
model = keras.Model(inputs=text_input, outputs=output)

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])



model.fit(train_set, validation_data=valid_set, epochs=10)

In [ ]:
# Save the trained model
model_path = "movie_sentiment_model.keras"
model.save(model_path)
print(f"Model saved to {model_path}")

Now, let's create a Flask application to serve a simple web page for movie review sentiment prediction. This app will load the saved model, provide an input form, and display the prediction result.

In [20]:
from flask import Flask, request, render_template_string
import tensorflow as tf
import tensorflow_hub as hub
import os

# It's important to define the custom layer in the scope where the model is loaded
# especially if it was defined in a notebook cell that might not be re-executed directly by the Flask app.
class USELayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.use_layer = hub.KerasLayer(
            "https://tfhub.dev/google/universal-sentence-encoder/4",
            trainable=True,
            name="USE_embedding"
        )

    def call(self, inputs):
        return self.use_layer(inputs)

app = Flask(__name__)

# Load the saved model
try:
    # Ensure TFHUB_CACHE_DIR is set if the model relies on TF-Hub assets
    os.environ["TFHUB_CACHE_DIR"] = "my_tfhub_cache"
    model = tf.keras.models.load_model(
        "movie_sentiment_model.keras",
        custom_objects={'USELayer': USELayer} # Important for custom layers
    )
    print("Model loaded successfully!")
except Exception as e:
    print(f"Error loading model: {e}")
    model = None # Set model to None if loading fails

# HTML template for the web page
HTML_TEMPLATE = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Movie Review Sentiment Analyzer</title>
    <style>
        body { font-family: Arial, sans-serif; margin: 20px; background-color: #f4f4f4; }
        .container { max-width: 600px; margin: auto; background: white; padding: 20px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); }
        h1 { color: #333; text-align: center; }
        textarea { width: 100%; padding: 10px; margin-bottom: 10px; border: 1px solid #ddd; border-radius: 4px; box-sizing: border-box; }
        button { background-color: #007bff; color: white; padding: 10px 15px; border: none; border-radius: 4px; cursor: pointer; font-size: 16px; }
        button:hover { background-color: #0056b3; }
        .result { margin-top: 20px; padding: 10px; background-color: #e9ecef; border-radius: 4px; }
        .positive { color: green; font-weight: bold; }
        .negative { color: red; font-weight: bold; }
    </style>
</head>
<body>
    <div class="container">
        <h1>Movie Review Sentiment Analyzer</h1>
        <form action="/predict" method="post">
            <label for="review">Enter your movie review:</label><br>
            <textarea id="review" name="review" rows="10" placeholder="Type your review here..."></textarea><br>
            <button type="submit">Analyze Sentiment</button>
        </form>
        {% if prediction %}
            <div class="result">
                <h2>Prediction:</h2>
                <p>The sentiment is: <span class="{{ prediction_class }}">{{ prediction }}</span></p>
            </div>
        {% endif %}
    </div>
</body>
</html>
"""

@app.route('/', methods=['GET'])
def home():
    return render_template_string(HTML_TEMPLATE)

@app.route('/predict', methods=['POST'])
def predict():
    if model is None:
        return "Error: Model not loaded. Please check the console for details."

    review_text = request.form['review']
    if not review_text:
        return render_template_string(HTML_TEMPLATE, prediction="Please enter a review.")

    # Make prediction
    # The model expects a batch of strings, so wrap the single review in a list
    predictions = model.predict([review_text])
    sentiment_score = predictions[0][0]

    if sentiment_score >= 0.5:
        sentiment = "Positive"
        prediction_class = "positive"
    else:
        sentiment = "Negative"
        prediction_class = "negative"

    return render_template_string(HTML_TEMPLATE, prediction=sentiment, prediction_class=prediction_class)

# To run the Flask app in Colab, we'll use a public URL provided by ngrok (or a similar service).
# If you're running this locally, you can just use app.run(debug=True).
# In Colab, you might need to install 'pyngrok' or use 'nest_asyncio'.

# The following code block is for running the app in Colab.
# You might need to install pyngrok first: !pip install pyngrok

from threading import Thread
from pyngrok import ngrok, conf
import nest_asyncio

# Apply nest_asyncio to allow asyncio to run in already running event loop
nest_asyncio.apply()

def start_ngrok():
    # This assumes you have an ngrok auth token set up for more stable tunnels
    # If not, you might get a temporary, rate-limited tunnel.
    # conf.get_default().auth_token = "YOUR_NGROK_AUTHTOKEN"
    public_url = ngrok.connect(5000)
    print(f" * ngrok tunnel: {public_url}")

# Start ngrok in a separate thread
ngrok_thread = Thread(target=start_ngrok)
ngrok_thread.daemon = True
ngrok_thread.start()

# Run the Flask app
# Setting debug=True is useful for development, but remove in production.
app.run(port=5000)


Error loading model: File not found: filepath=movie_sentiment_model.keras. Please ensure the file is an accessible `.keras` zip file.
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit


ERROR:pyngrok.process.ngrok:t=2026-04-14T08:53:13+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-04-14T08:53:13+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-04-14T08:53:13+0000 lvl=eror msg="terminating with error" obj=app err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your aut